# ALS Causal Knowledge Graph: Community Structure and Multi-Biomarker Hypothesis Testing

This notebook loads a pre-built ALS Causal Knowledge Graph (CKG) and tests four hypotheses about
biomarker community structure. It reproduces the community-detection analysis reported in the
accompanying paper. Each section prints intermediate output so you can validate progress as you go.

**Hypotheses tested (matching the paper):**
- **Hypothesis I** — Biomarker communities are more densely interconnected with each other than with genetic communities.
- **Hypothesis II** — Neurofilament (NfL) nodes occupy central, hub-like positions in the graph.
- **Hypothesis III** — The Louvain community structure is stable across random seeds and resolution parameters.
- **Hypothesis IV** — Top-degree nodes in the CSF communities correspond to clinically established ALS biomarkers.

---
**Note on the data.** The CKG was built from 35 ALS CSF proteomics papers. Results are a pilot-scale
analysis within that literature slice, not a comprehensive ALS map. Statistical tests have limited power
and should be read as directional / hypothesis-generating.

**Before running:** edit the single file path in *Step 1* to point to your graph pickle. All output
files (figures and CSVs) are written to the current working directory. See the README for setup and for
exactly what the graph pickle must contain.

## Step 0: Environment Setup

Install the dependencies once with the next cell, or create the environment from `requirements.txt`
as described in the README. All analysis runs on CPU via NetworkX, which is fast for a graph of this
size; no GPU is required.

In [ ]:
# Install required libraries (optional: skip if you built the environment from requirements.txt).
# python-louvain installs under the pip name "python-louvain" but imports as "community".
import subprocess, sys

packages = [
    'networkx>=2.8',
    'python-louvain>=0.16',
    'matplotlib>=3.6,<3.9',   # get_cmap(name, N) is used; it was removed in matplotlib 3.9
    'seaborn>=0.12',
    'pandas>=1.5,<3',
    'numpy>=1.23,<2',
    'scipy>=1.9',
    'scikit-learn>=1.1',      # provides normalized_mutual_info_score (Hypothesis III)
    'tabulate>=0.9',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages installed.')

In [ ]:
# Core imports
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import community as community_louvain   # python-louvain package (pip install python-louvain)
from collections import Counter, defaultdict
from scipy import stats
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Imports complete.')

## Step 1: Load the Graph

**Edit the single file path below** to point to your CKG pickle. Only one file is required: the final
graph, containing node names and a numeric edge-weight attribute, and ideally a per-node `community`
attribute. The loader auto-detects whether the pickle holds a NetworkX graph or a dict and reports what
it finds. See the README for the exact schema the pickle must contain.

In [ ]:
# ============================================================
# EDIT THIS PATH to point to your graph pickle (the only required input).
# ============================================================
PKL_GRAPH = './ckg_louvain.pkl'      # <-- CHANGE to your filename / path
# ============================================================

def load_pkl(path):
    """Load a pickle file and report its type and basic shape."""
    if not os.path.exists(path):
        print(f'  [ERROR] File not found: {path}')
        print('  Edit PKL_GRAPH above to the correct path, then re-run this cell.')
        return None
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    print(f'  Loaded: {path}')
    print(f'  Type  : {type(obj)}')
    if isinstance(obj, nx.Graph):
        print(f'  Nodes : {obj.number_of_nodes():,}')
        print(f'  Edges : {obj.number_of_edges():,}')
        if obj.number_of_nodes() > 0:
            sample_node = list(obj.nodes(data=True))[0]
            print(f'  Sample node keys: {list(sample_node[1].keys())}')
    elif isinstance(obj, dict):
        print(f'  Dict keys (first 10): {list(obj.keys())[:10]}')
    elif isinstance(obj, (list, tuple)):
        print(f'  Length: {len(obj)}')
        print(f'  First element type: {type(obj[0])}')
    print()
    return obj

print('Loading graph pickle...\n')
graph_obj = load_pkl(PKL_GRAPH)

### Step 1b: Normalize to NetworkX Graph

This cell adapts to whatever structure your pkl files contain. If they are already NetworkX graphs it passes them through. If they are dicts it attempts to reconstruct the graph.

In [ ]:
def extract_graph(obj, label='graph'):
    """
    Try to extract a NetworkX graph from various pkl structures.
    Handles: nx.Graph directly, dict with a graph under a known key,
    or a dict that is itself an adjacency structure.
    Returns a NetworkX graph or None.
    """
    if obj is None:
        return None

    # Already a graph
    if isinstance(obj, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        print(f'{label}: Already a NetworkX graph ({type(obj).__name__})')
        return obj

    # Dict that contains a graph object under a key
    if isinstance(obj, dict):
        for key in ['graph', 'G', 'ckg', 'network', 'kg']:
            if key in obj and isinstance(obj[key], nx.Graph):
                print(f'{label}: Found NetworkX graph under dict key "{key}"')
                return obj[key]

        # Dict that IS the adjacency structure (node -> {neighbor: attr})
        try:
            G = nx.from_dict_of_dicts(obj)
            print(f'{label}: Reconstructed graph from adjacency dict ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)')
            return G
        except Exception:
            pass

        print(f'{label}: Dict with keys {list(obj.keys())[:10]} - manual extraction needed.')
        print('       Please inspect the dict and assign G manually below.')
        return None

    print(f'{label}: Unrecognized type {type(obj)}. Assign G manually.')
    return None

G = extract_graph(graph_obj, label='CKG')

# ============================================================
# MANUAL OVERRIDE: if auto-extraction failed, assign G here, e.g.:
#   G = graph_obj['your_graph_key']
# ============================================================
assert G is not None, 'No valid graph found. Check PKL_GRAPH and the manual override above.'

print(f'\nWorking graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Graph type: {type(G).__name__}')

### Step 1c: Inspect Node Attributes

This cell prints the attribute fields available on nodes and edges so you can verify the data schema matches what the paper describes.

In [ ]:
# Print first 3 nodes with all attributes
print('=== Sample node attributes ===')
for i, (node, attrs) in enumerate(G.nodes(data=True)):
    print(f'\nNode: {node}')
    for k, v in attrs.items():
        if isinstance(v, np.ndarray):
            print(f'  {k}: ndarray shape={v.shape}')
        elif isinstance(v, list) and len(v) > 4:
            print(f'  {k}: list len={len(v)} first={v[:2]}')
        else:
            print(f'  {k}: {v}')
    if i >= 2:
        break

print('\n=== Sample edge attributes ===')
for i, (u, v, attrs) in enumerate(G.edges(data=True)):
    print(f'\nEdge: {u} -> {v}')
    for k, val in attrs.items():
        if isinstance(val, np.ndarray):
            print(f'  {k}: ndarray shape={val.shape}')
        elif isinstance(val, list) and len(val) > 4:
            print(f'  {k}: list len={len(val)}')
        else:
            print(f'  {k}: {val}')
    if i >= 2:
        break

### Step 1d: Build Community Lookup Table

This cell extracts community assignments from node attributes. If your Louvain pkl already has community labels stored per node, it uses those. Otherwise it re-runs Louvain on the fly. Re-running is fast (under 60 seconds for your graph size).

In [ ]:
# ============================================================
# EDIT if your community label attribute has a different name
# Common names: 'community', 'cluster', 'louvain_community', 'group'
COMMUNITY_ATTR = 'community'
# ============================================================

# Check if community labels already exist in node attributes
sample_attrs = dict(list(G.nodes(data=True))[0][1])
has_community_attr = COMMUNITY_ATTR in sample_attrs

if has_community_attr:
    print(f'Community labels found in node attribute "{COMMUNITY_ATTR}"')
    node_community = {node: attrs.get(COMMUNITY_ATTR) for node, attrs in G.nodes(data=True)}
else:
    print(f'No "{COMMUNITY_ATTR}" attribute found. Re-running Louvain with gamma=1.0...')
    # Convert to undirected for Louvain if needed
    G_undirected = G.to_undirected() if G.is_directed() else G

    # Use edge weight if available
    weight_key = None
    sample_edge = list(G_undirected.edges(data=True))
    if sample_edge:
        edge_attrs = sample_edge[0][2]
        for candidate in ['weight', 'w_norm', 'edge_confidence', 'confidence', 'tcvs']:
            if candidate in edge_attrs:
                weight_key = candidate
                break

    print(f'Using edge weight attribute: {weight_key}')
    node_community = community_louvain.best_partition(G_undirected, weight=weight_key, resolution=1.0, random_state=42)
    # Store back in graph
    nx.set_node_attributes(G, node_community, name='community')

# Summarize communities
community_counts = Counter(node_community.values())
n_communities = len(community_counts)
print(f'\nTotal communities detected: {n_communities}')
print(f'Largest community: {max(community_counts.values())} nodes')
print(f'Smallest community: {min(community_counts.values())} nodes')
print(f'Median community size: {np.median(list(community_counts.values())):.0f} nodes')

### Step 1e: Label Communities and Identify Biomarker-Relevant Groups

This cell labels communities by their most frequent node terms and maps them to the biological categories from the paper. You may need to adjust the COMMUNITY_LABELS dict based on what you see in the output.

In [ ]:
def get_community_top_nodes(G, node_community, n=10):
    """For each community, return top-n nodes by degree."""
    comm_nodes = defaultdict(list)
    for node, comm in node_community.items():
        comm_nodes[comm].append(node)

    result = {}
    for comm, nodes in comm_nodes.items():
        degrees = {n: G.degree(n) for n in nodes}
        top = sorted(degrees.items(), key=lambda x: -x[1])[:n]
        result[comm] = {
            'size': len(nodes),
            'top_nodes': [t[0] for t in top],
            'top_degrees': [t[1] for t in top],
            'nodes': nodes
        }
    return result

comm_summary = get_community_top_nodes(G, node_community)

# Print community summary table
print('Community summary (sorted by size):')
print()
rows = []
for comm_id, info in sorted(comm_summary.items(), key=lambda x: -x[1]['size']):
    top3 = ', '.join(str(n) for n in info['top_nodes'][:3])
    rows.append([comm_id, info['size'], top3])

print(tabulate(rows, headers=['Community ID', 'Size', 'Top 3 Nodes by Degree'], tablefmt='grid'))

In [ ]:
# ============================================================
# EDIT THIS based on the community summary output above.
#
# Map community IDs to biological labels.
# Based on the paper, the expected communities are:
#   Markers (biomarker ecosystem)
#   ALS (core disease)
#   progression (disease advancement)
#   APOE (lipid/neuroinflammation)
#   patients (clinical)
#   C9orf72 (genetic)
#   familial ALS (genetic/SOD1)
#   CSF (fluid biomarkers)
#
# Look at the top nodes from the output above and fill in the IDs.
# Example: if community 0 has top nodes 'Markers', 'Biomarker', fill: 0: 'Markers'
# ============================================================

COMMUNITY_LABELS = {
    # Fill these in after reviewing the summary above
    # 0: 'Markers',
    # 1: 'ALS',
    # 2: 'progression',
    # 3: 'APOE',
    # 4: 'patients',
    # 5: 'C9orf72',
    # 6: 'familial ALS',
    # 7: 'CSF',
}

# If you have not filled in the labels yet, auto-generate them from top node names
if not COMMUNITY_LABELS:
    print('COMMUNITY_LABELS not filled in. Auto-generating from top node names...')
    for comm_id, info in sorted(comm_summary.items(), key=lambda x: -x[1]['size']):
        label = str(info['top_nodes'][0]) if info['top_nodes'] else f'Community_{comm_id}'
        # Truncate long labels
        COMMUNITY_LABELS[comm_id] = label[:30]
    print('Auto-generated labels. Please review and edit COMMUNITY_LABELS manually for accuracy.')

# Biological category grouping for hypothesis tests
# EDIT: map community IDs to biological supertype
# 'biomarker' = Markers, APOE, CSF communities
# 'genetic'   = C9orf72, familial ALS communities
# 'clinical'  = patients, progression communities
# 'core'      = ALS core community

# Auto-assign based on label keywords
BIOMARKER_KEYWORDS = ['marker', 'biomarker', 'apoe', 'csf', 'nfl', 'neurofilament', 'fluid']
GENETIC_KEYWORDS   = ['c9orf72', 'sod1', 'familial', 'genetic', 'mutation', 'gene']
CLINICAL_KEYWORDS  = ['patient', 'progression', 'clinical', 'survival', 'onset']

community_supertype = {}
for comm_id, label in COMMUNITY_LABELS.items():
    label_lower = label.lower()
    if any(kw in label_lower for kw in BIOMARKER_KEYWORDS):
        community_supertype[comm_id] = 'biomarker'
    elif any(kw in label_lower for kw in GENETIC_KEYWORDS):
        community_supertype[comm_id] = 'genetic'
    elif any(kw in label_lower for kw in CLINICAL_KEYWORDS):
        community_supertype[comm_id] = 'clinical'
    else:
        community_supertype[comm_id] = 'other'

print('\nCommunity supertype assignments:')
for comm_id, supertype in sorted(community_supertype.items()):
    label = COMMUNITY_LABELS.get(comm_id, f'Community_{comm_id}')
    size = comm_summary[comm_id]['size']
    print(f'  Community {comm_id} ({label}, n={size}): {supertype}')

print('\nIf supertypes look wrong, edit COMMUNITY_LABELS and re-run this cell.')

## Step 2: Basic Graph Visualization

A community-colored network plot to confirm the graph loaded correctly and communities look reasonable.

In [ ]:
def plot_community_graph(G, node_community, community_labels, max_nodes=500, title='CKG Community Structure'):
    """
    Plot community-colored graph. Samples nodes if graph is large.
    """
    # Sample for visibility if needed
    all_nodes = list(G.nodes())
    if len(all_nodes) > max_nodes:
        print(f'Graph has {len(all_nodes)} nodes. Sampling {max_nodes} for visualization...')
        # Sample proportionally from each community
        sampled = []
        comm_groups = defaultdict(list)
        for n in all_nodes:
            comm_groups[node_community.get(n, -1)].append(n)
        for comm, nodes in comm_groups.items():
            k = max(1, int(max_nodes * len(nodes) / len(all_nodes)))
            sampled.extend(nodes[:k])
        G_sub = G.subgraph(sampled)
    else:
        G_sub = G

    # Assign colors
    unique_comms = sorted(set(node_community.values()))
    try:
        cmap = plt.cm.get_cmap('tab20', len(unique_comms))       # matplotlib < 3.9
    except AttributeError:
        import matplotlib as mpl
        cmap = mpl.colormaps['tab20'].resampled(len(unique_comms))  # matplotlib >= 3.9
    comm_color = {c: cmap(i) for i, c in enumerate(unique_comms)}

    node_colors = [comm_color[node_community.get(n, -1)] for n in G_sub.nodes()]
    node_sizes  = [max(20, G.degree(n) * 2) for n in G_sub.nodes()]

    fig, ax = plt.subplots(figsize=(14, 10))

    # Layout
    try:
        pos = nx.spring_layout(G_sub, k=0.5, seed=42)
    except Exception:
        pos = nx.kamada_kawai_layout(G_sub)

    # Draw
    nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors, node_size=node_sizes,
                           alpha=0.85, ax=ax)
    nx.draw_networkx_edges(G_sub, pos, edge_color='#cccccc', alpha=0.3,
                           arrows=False, ax=ax)

    # Label top 3 nodes per community by degree
    labeled = set()
    for comm in unique_comms:
        comm_nodes_sub = [n for n in G_sub.nodes() if node_community.get(n) == comm]
        if not comm_nodes_sub:
            continue
        top = sorted(comm_nodes_sub, key=lambda n: G.degree(n), reverse=True)[:2]
        for n in top:
            labeled.add(n)

    label_dict = {n: str(n)[:20] for n in labeled}
    nx.draw_networkx_labels(G_sub, pos, labels=label_dict, font_size=6, ax=ax)

    # Legend
    patches = []
    for comm in unique_comms:
        label = community_labels.get(comm, f'Community {comm}')
        size  = comm_summary[comm]['size'] if comm in comm_summary else '?'
        patches.append(mpatches.Patch(color=comm_color[comm], label=f'{label} (n={size})'))

    ax.legend(handles=patches, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8,
              frameon=True, title='Communities')
    ax.set_title(title, fontsize=13, pad=12)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('fig_community_graph.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('Saved: fig_community_graph.png')

plot_community_graph(G, node_community, COMMUNITY_LABELS)

## Hypothesis I: Biomarker Communities Are More Densely Interconnected With Each Other Than With Genetic Communities

**What we measure:** Cross-community edge density between biomarker-type communities versus between biomarker and genetic communities. We compute a simple ratio and test with a permutation approach.

In [ ]:
def compute_cross_community_edges(G, node_community, community_supertype):
    """
    Count edges between pairs of communities, categorized by supertype pairs.
    Returns a DataFrame with (comm_A, comm_B, n_edges, type_A, type_B) rows.
    """
    edge_counts = defaultdict(int)
    total_edges = 0

    for u, v in G.edges():
        cu = node_community.get(u, -1)
        cv = node_community.get(v, -1)
        if cu != cv:  # only cross-community edges
            key = (min(cu, cv), max(cu, cv))
            edge_counts[key] += 1
            total_edges += 1

    rows = []
    for (ca, cb), count in edge_counts.items():
        ta = community_supertype.get(ca, 'other')
        tb = community_supertype.get(cb, 'other')
        label_a = COMMUNITY_LABELS.get(ca, f'C{ca}')
        label_b = COMMUNITY_LABELS.get(cb, f'C{cb}')
        size_a  = comm_summary[ca]['size'] if ca in comm_summary else 1
        size_b  = comm_summary[cb]['size'] if cb in comm_summary else 1
        # Normalized density = edges / (size_a * size_b)
        density = count / (size_a * size_b)
        rows.append({
            'comm_A': ca, 'comm_B': cb,
            'label_A': label_a, 'label_B': label_b,
            'type_A': ta, 'type_B': tb,
            'edge_count': count,
            'density': density,
            'pair_type': '-'.join(sorted([ta, tb]))
        })

    df = pd.DataFrame(rows)
    print(f'Total cross-community edges: {total_edges:,}')
    return df

cross_df = compute_cross_community_edges(G, node_community, community_supertype)
print(f'Unique community pairs with cross-edges: {len(cross_df)}')
print()

# Summarize by pair type
pair_summary = cross_df.groupby('pair_type').agg(
    n_pairs=('edge_count', 'count'),
    total_edges=('edge_count', 'sum'),
    mean_density=('density', 'mean'),
    median_density=('density', 'median')
).reset_index()

print('Cross-community edge summary by supertype pair:')
print(tabulate(pair_summary.round(6), headers='keys', tablefmt='grid', showindex=False))

In [ ]:
# Plot Hypothesis I: cross-community edge density heatmap

unique_comms = sorted(set(node_community.values()))
n_comm = len(unique_comms)
comm_index = {c: i for i, c in enumerate(unique_comms)}

# Build density matrix
density_matrix = np.zeros((n_comm, n_comm))
count_matrix   = np.zeros((n_comm, n_comm))

for _, row in cross_df.iterrows():
    i = comm_index[row['comm_A']]
    j = comm_index[row['comm_B']]
    density_matrix[i, j] = row['density']
    density_matrix[j, i] = row['density']
    count_matrix[i, j]   = row['edge_count']
    count_matrix[j, i]   = row['edge_count']

comm_tick_labels = [COMMUNITY_LABELS.get(c, f'C{c}')[:15] for c in unique_comms]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Density heatmap
sns.heatmap(density_matrix, ax=axes[0],
            xticklabels=comm_tick_labels, yticklabels=comm_tick_labels,
            cmap='YlOrRd', linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Edge density'})
axes[0].set_title('Cross-community edge density\n(edges / node_A x node_B)', fontsize=11)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0, labelsize=8)

# Edge count heatmap
sns.heatmap(count_matrix, ax=axes[1],
            xticklabels=comm_tick_labels, yticklabels=comm_tick_labels,
            cmap='Blues', linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Edge count'})
axes[1].set_title('Cross-community edge counts', fontsize=11)
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0, labelsize=8)

plt.suptitle('Hypothesis I: Cross-Community Connectivity', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_h1_cross_community.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_h1_cross_community.png')

In [ ]:
# Statistical comparison: biomarker-biomarker vs biomarker-genetic density
bio_bio    = cross_df[cross_df['pair_type'] == 'biomarker-biomarker']['density'].values
bio_gen    = cross_df[cross_df['pair_type'] == 'biomarker-genetic']['density'].values
bio_other  = cross_df[(cross_df['type_A'] == 'biomarker') | (cross_df['type_B'] == 'biomarker')]['density'].values

print('=== Hypothesis I: Statistical Summary ===')
print()
print(f'Biomarker-biomarker pairs: n={len(bio_bio)}')
if len(bio_bio) > 0:
    print(f'  Mean density  : {bio_bio.mean():.6f}')
    print(f'  Median density: {np.median(bio_bio):.6f}')

print(f'\nBiomarker-genetic pairs: n={len(bio_gen)}')
if len(bio_gen) > 0:
    print(f'  Mean density  : {bio_gen.mean():.6f}')
    print(f'  Median density: {np.median(bio_gen):.6f}')

# Mann-Whitney U test (non-parametric, appropriate for small samples)
if len(bio_bio) >= 3 and len(bio_gen) >= 3:
    stat, p = stats.mannwhitneyu(bio_bio, bio_gen, alternative='greater')
    print(f'\nMann-Whitney U (bio-bio > bio-genetic): stat={stat:.1f}, p={p:.4f}')
    if p < 0.05:
        print('Result: SUPPORTS Hypothesis I (p < 0.05)')
    elif p < 0.10:
        print('Result: Marginal support for Hypothesis I (0.05 < p < 0.10)')
    else:
        print('Result: Hypothesis I NOT supported. Report descriptively, not as statistical finding.')
else:
    print('\nInsufficient pairs for statistical test. Report descriptively.')
    if len(bio_bio) > 0 and len(bio_gen) > 0:
        ratio = bio_bio.mean() / bio_gen.mean() if bio_gen.mean() > 0 else float('inf')
        print(f'Descriptive ratio (bio-bio mean / bio-genetic mean): {ratio:.2f}x')

print()
print('IMPORTANT CAVEAT: With 35 input papers, small community pairs may have few cross-edges.')
print('Report the direction of the finding rather than making strong statistical claims.')

## Hypothesis II: NfL and Neurofilament Markers as Hub Nodes Bridging Multiple Communities

**What we measure:** Betweenness centrality for neurofilament-related nodes versus all other nodes. High betweenness means a node lies on many shortest paths between other nodes, making it a structural bridge.

In [ ]:
# Compute betweenness and degree centrality
# Note: betweenness on large graphs can be slow. We use approximate k-sample version if needed.

print('Computing degree centrality...')
degree_centrality = dict(G.degree())

n_nodes = G.number_of_nodes()
if n_nodes > 2000:
    print(f'Graph has {n_nodes} nodes. Using approximate betweenness (k=500 samples)...')
    betweenness = nx.betweenness_centrality(G, k=min(500, n_nodes), normalized=True, seed=42)
else:
    print(f'Computing exact betweenness for {n_nodes} nodes...')
    betweenness = nx.betweenness_centrality(G, normalized=True)

print('Done.')

# Build centrality DataFrame
centrality_df = pd.DataFrame({
    'node': list(degree_centrality.keys()),
    'degree': list(degree_centrality.values()),
    'betweenness': [betweenness.get(n, 0) for n in degree_centrality.keys()],
    'community': [node_community.get(n, -1) for n in degree_centrality.keys()],
    'community_label': [COMMUNITY_LABELS.get(node_community.get(n, -1), f'C{node_community.get(n,-1)}') 
                        for n in degree_centrality.keys()]
})

# Flag neurofilament / NfL related nodes
NFL_KEYWORDS = [
    'nfl', 'neurofilament', 'nfl', 'pfnh', 'pnfh', 'neurofilament light',
    'neurofilament heavy', 'nf-l', 'nf-h', 'neurofilament chain'
]

def is_nfl_node(node_name):
    node_str = str(node_name).lower()
    return any(kw in node_str for kw in NFL_KEYWORDS)

centrality_df['is_nfl'] = centrality_df['node'].apply(is_nfl_node)

nfl_nodes = centrality_df[centrality_df['is_nfl']]
print(f'\nNeurofilament-related nodes found: {len(nfl_nodes)}')
if len(nfl_nodes) > 0:
    print(tabulate(nfl_nodes[['node', 'degree', 'betweenness', 'community_label']]
                   .sort_values('betweenness', ascending=False).head(15),
                   headers='keys', tablefmt='grid', showindex=False, floatfmt='.6f'))
else:
    print('No NfL nodes found with current keywords. Check your node naming.')
    print('Top 20 nodes by betweenness:')
    print(tabulate(centrality_df.nlargest(20, 'betweenness')[['node', 'degree', 'betweenness', 'community_label']],
                   headers='keys', tablefmt='grid', showindex=False, floatfmt='.6f'))

In [ ]:
# Plot Hypothesis II: Betweenness vs Degree scatter, highlighting NfL nodes
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: betweenness vs degree
ax = axes[0]
other_nodes  = centrality_df[~centrality_df['is_nfl']]
nfl_nodes_df = centrality_df[centrality_df['is_nfl']]

ax.scatter(other_nodes['degree'], other_nodes['betweenness'],
           c='#a0aec0', alpha=0.4, s=15, label='Other nodes', zorder=1)
if len(nfl_nodes_df) > 0:
    ax.scatter(nfl_nodes_df['degree'], nfl_nodes_df['betweenness'],
               c='#e53e3e', s=80, zorder=3, label='Neurofilament nodes', edgecolors='#822727')
    for _, row in nfl_nodes_df.iterrows():
        label = str(row['node'])[:20]
        ax.annotate(label, (row['degree'], row['betweenness']),
                    xytext=(5, 5), textcoords='offset points', fontsize=7, color='#822727')

ax.set_xlabel('Degree', fontsize=11)
ax.set_ylabel('Betweenness centrality', fontsize=11)
ax.set_title('Hypothesis II: Neurofilament hub analysis', fontsize=11)
ax.legend(fontsize=9)

# Rank plot: top 30 nodes by betweenness
ax2 = axes[1]
top30 = centrality_df.nlargest(30, 'betweenness').reset_index(drop=True)
colors = ['#e53e3e' if r else '#4a5568' for r in top30['is_nfl']]
bars = ax2.barh(range(len(top30)), top30['betweenness'], color=colors)
ax2.set_yticks(range(len(top30)))
ax2.set_yticklabels([str(n)[:25] for n in top30['node']], fontsize=7)
ax2.invert_yaxis()
ax2.set_xlabel('Betweenness centrality', fontsize=11)
ax2.set_title('Top 30 nodes by betweenness centrality', fontsize=11)
red_patch   = mpatches.Patch(color='#e53e3e', label='Neurofilament')
gray_patch  = mpatches.Patch(color='#4a5568', label='Other')
ax2.legend(handles=[red_patch, gray_patch], fontsize=9)

plt.tight_layout()
plt.savefig('fig_h2_betweenness.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_h2_betweenness.png')

In [ ]:
# Statistical test: are NfL nodes significantly higher in betweenness?
print('=== Hypothesis II: Statistical Summary ===')
nfl_btw   = centrality_df[centrality_df['is_nfl']]['betweenness'].values
other_btw = centrality_df[~centrality_df['is_nfl']]['betweenness'].values

if len(nfl_btw) >= 2:
    stat, p = stats.mannwhitneyu(nfl_btw, other_btw, alternative='greater')
    print(f'NfL nodes: n={len(nfl_btw)}, mean betweenness={nfl_btw.mean():.6f}')
    print(f'Other nodes: n={len(other_btw)}, mean betweenness={other_btw.mean():.6f}')
    print(f'Ratio (NfL / other): {nfl_btw.mean() / other_btw.mean():.2f}x')
    print(f'Mann-Whitney U (NfL > other): stat={stat:.1f}, p={p:.4f}')
    if p < 0.05:
        print('Result: SUPPORTS Hypothesis II')
    else:
        print('Result: Not statistically significant. Report the ratio descriptively.')
elif len(nfl_btw) == 1:
    pctile = stats.percentileofscore(other_btw, nfl_btw[0])
    print(f'Only 1 NfL node found. Betweenness percentile vs all other nodes: {pctile:.1f}th')
    if pctile > 80:
        print('The NfL node ranks in the top 20% of the graph by betweenness centrality.')
else:
    print('No NfL nodes found. Hypothesis II cannot be tested. Check node naming in NFL_KEYWORDS.')

## Hypothesis III: Community Structure Stability Across Random Seeds and Resolution Parameters

**What we measure:** Normalized Mutual Information (NMI) between Louvain partitions at different random seeds and resolution parameters. NMI = 1.0 means identical partitions; NMI > 0.8 suggests robust community structure.

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

# Convert to undirected for Louvain
G_undir = G.to_undirected() if G.is_directed() else G

# Detect weight key
weight_key = None
if G_undir.number_of_edges() > 0:
    sample_edge_attrs = list(G_undir.edges(data=True))[0][2]
    for candidate in ['weight', 'w_norm', 'edge_confidence', 'confidence']:
        if candidate in sample_edge_attrs:
            weight_key = candidate
            break

print(f'Using edge weight: {weight_key}')
print('Running Louvain with 5 random seeds at resolution=1.0...')

# Reference partition (seed=42)
ref_partition = community_louvain.best_partition(G_undir, weight=weight_key, resolution=1.0, random_state=42)
ref_labels = [ref_partition[n] for n in sorted(G_undir.nodes())]

# Seed stability
seeds = [0, 1, 7, 13, 99]
seed_results = []
for seed in seeds:
    part = community_louvain.best_partition(G_undir, weight=weight_key, resolution=1.0, random_state=seed)
    labels = [part[n] for n in sorted(G_undir.nodes())]
    nmi = normalized_mutual_info_score(ref_labels, labels)
    n_comms = len(set(part.values()))
    seed_results.append({'seed': seed, 'n_communities': n_comms, 'NMI_vs_seed42': nmi})

seed_df = pd.DataFrame(seed_results)
print('\nSeed stability results:')
print(tabulate(seed_df.round(4), headers='keys', tablefmt='grid', showindex=False))

# Resolution stability
print('\nRunning Louvain at different resolution parameters (seed=42)...')
resolutions = [0.6, 0.8, 1.0, 1.2, 1.5]
res_results = []
for gamma in resolutions:
    part = community_louvain.best_partition(G_undir, weight=weight_key, resolution=gamma, random_state=42)
    labels = [part[n] for n in sorted(G_undir.nodes())]
    nmi = normalized_mutual_info_score(ref_labels, labels)
    n_comms = len(set(part.values()))
    res_results.append({'resolution': gamma, 'n_communities': n_comms, 'NMI_vs_gamma1.0': nmi})

res_df = pd.DataFrame(res_results)
print('\nResolution stability results:')
print(tabulate(res_df.round(4), headers='keys', tablefmt='grid', showindex=False))

In [ ]:
# Plot Hypothesis III
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Seed stability
axes[0].bar(range(len(seed_df)), seed_df['NMI_vs_seed42'], color='#2b6cb0', alpha=0.8)
axes[0].axhline(0.8, color='#e53e3e', linestyle='--', linewidth=1.5, label='NMI=0.8 threshold')
axes[0].set_xticks(range(len(seed_df)))
axes[0].set_xticklabels([f'Seed {s}' for s in seed_df['seed']])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Normalized Mutual Information')
axes[0].set_title('Seed stability\n(vs seed=42 reference partition)')
axes[0].legend(fontsize=9)
for i, row in seed_df.iterrows():
    axes[0].text(i, row['NMI_vs_seed42'] + 0.01, f"{row['NMI_vs_seed42']:.3f}",
                ha='center', fontsize=8)

# Resolution stability
axes[1].plot(res_df['resolution'], res_df['NMI_vs_gamma1.0'], 'o-', color='#2b6cb0', linewidth=2, markersize=8)
axes[1].axhline(0.8, color='#e53e3e', linestyle='--', linewidth=1.5, label='NMI=0.8 threshold')
# Secondary axis for n_communities
ax2_r = axes[1].twinx()
ax2_r.plot(res_df['resolution'], res_df['n_communities'], 's--', color='#718096', linewidth=1.5, markersize=7, alpha=0.7)
ax2_r.set_ylabel('Number of communities', color='#718096', fontsize=10)
axes[1].set_xlabel('Resolution parameter (gamma)')
axes[1].set_ylabel('NMI vs gamma=1.0')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Resolution stability\n(solid=NMI, dashed=n_communities)')
axes[1].legend(fontsize=9)

plt.suptitle('Hypothesis III: Louvain Community Stability', fontsize=13)
plt.tight_layout()
plt.savefig('fig_h3_stability.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_h3_stability.png')

# Summary interpretation
mean_seed_nmi = seed_df['NMI_vs_seed42'].mean()
print(f'\nMean NMI across seeds: {mean_seed_nmi:.4f}')
if mean_seed_nmi > 0.85:
    print('Community structure is highly stable across random seeds. Strong result for the paper.')
elif mean_seed_nmi > 0.70:
    print('Community structure is moderately stable. Report with appropriate caveats for pilot study.')
else:
    print('Community structure is sensitive to seed. Major caveat: communities may be artifact of optimization.')
    print('Consider reporting only the most consistent communities (present across all seeds).')

## Hypothesis IV: Top CSF Community Nodes Correspond to Clinically Established ALS Biomarkers

**What we measure:** We rank nodes in the CSF community by degree centrality and check whether the top-ranked nodes match the established ALS biomarker literature. This is a validation check, not a statistical test.

In [ ]:
# Established ALS CSF biomarkers from the literature
# References: Verde et al. (2019), Thompson et al. (2019),
#             Benatar et al. (2020), Steinacker et al. (2018)
ESTABLISHED_BIOMARKERS = [
    # Neurofilaments
    'nfl', 'neurofilament light', 'neurofilament light chain', 'nfl csf',
    'nf-l', 'neurofilament',
    'pnfh', 'pfnh', 'phosphorylated neurofilament heavy',
    'neurofilament heavy', 'nfh',
    # Chitinases
    'chit1', 'chitotriosidase', 'chitotriosidase-1', 'chitinase',
    'chi3l1', 'ykl-40', 'ykl40',
    'chi3l2', 'ykl-39',
    # Neuroinflammation
    'trem2', 'apoe', 'apoe4', 'apoe e4',
    'complement', 'c1q',
    # Others
    'tdp-43', 'tdp43',
    'nptx2', 'neuronal pentraxin',
    'c9orf72',
    'sod1',
    'tofersen',
    'riluzole',
    'apoc1',
]

def is_established_biomarker(node_name):
    node_str = str(node_name).lower()
    return any(bm in node_str for bm in ESTABLISHED_BIOMARKERS)

# Find CSF community
CSF_KEYWORDS = ['csf', 'cerebrospinal', 'fluid']
csf_comm_ids = [
    cid for cid, label in COMMUNITY_LABELS.items()
    if any(kw in label.lower() for kw in CSF_KEYWORDS)
]

if not csf_comm_ids:
    print('No CSF community found by keyword.')
    print('Using top 3 biomarker communities instead.')
    csf_comm_ids = [cid for cid, st in community_supertype.items() if st == 'biomarker'][:3]

print(f'CSF/biomarker community IDs for Hypothesis IV: {csf_comm_ids}')
csf_nodes = [n for n, c in node_community.items() if c in csf_comm_ids]
print(f'Total nodes in these communities: {len(csf_nodes)}')

# Rank by degree
csf_df = pd.DataFrame([
    {
        'node': n,
        'degree': G.degree(n),
        'betweenness': betweenness.get(n, 0),
        'community': COMMUNITY_LABELS.get(node_community.get(n), '?'),
        'is_established': is_established_biomarker(n)
    }
    for n in csf_nodes
]).sort_values('degree', ascending=False)

top_csf = csf_df.head(30)
print(f'\nTop 30 CSF/biomarker community nodes by degree:')
print(tabulate(
    top_csf[['node', 'degree', 'betweenness', 'community', 'is_established']].round(6),
    headers=['Node', 'Degree', 'Betweenness', 'Community', 'Established Biomarker?'],
    tablefmt='grid', showindex=False
))

# Precision in top-N
for k in [5, 10, 20]:
    top_k = csf_df.head(k)
    precision = top_k['is_established'].sum() / k
    print(f'Precision@{k}: {top_k["is_established"].sum()}/{k} = {precision:.2%}')

overall_precision = csf_df['is_established'].sum() / len(csf_df)
print(f'\nOverall fraction of established biomarkers in community: {csf_df["is_established"].sum()}/{len(csf_df)} = {overall_precision:.2%}')
print('\nNote: A low match may mean your node names use different terminology.')
print('Check top nodes manually against ESTABLISHED_BIOMARKERS list and adjust as needed.')

In [ ]:
# Plot Hypothesis IV
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Bar chart: top 30 CSF nodes by degree, color by established status
top_n = min(30, len(csf_df))
top_csf_plot = csf_df.head(top_n).reset_index(drop=True)
colors = ['#2b6cb0' if x else '#a0aec0' for x in top_csf_plot['is_established']]

axes[0].barh(range(top_n), top_csf_plot['degree'], color=colors)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels([str(n)[:30] for n in top_csf_plot['node']], fontsize=7)
axes[0].invert_yaxis()
axes[0].set_xlabel('Degree')
axes[0].set_title(f'Top {top_n} CSF/Biomarker community nodes\nby degree centrality')
blue_patch = mpatches.Patch(color='#2b6cb0', label='Established ALS biomarker')
gray_patch  = mpatches.Patch(color='#a0aec0', label='Other')
axes[0].legend(handles=[blue_patch, gray_patch], fontsize=9)

# Precision@K bar chart
ks = list(range(5, min(51, len(csf_df)+1), 5))
precisions = [csf_df.head(k)['is_established'].mean() for k in ks]
axes[1].bar(ks, precisions, color='#2b6cb0', alpha=0.8, width=4)
axes[1].axhline(overall_precision, color='#e53e3e', linestyle='--',
                linewidth=1.5, label=f'Overall community rate ({overall_precision:.1%})')
axes[1].set_xlabel('Top-K cutoff')
axes[1].set_ylabel('Fraction matching established biomarkers')
axes[1].set_title('Hypothesis IV: Biomarker precision\nin top-K nodes by degree')
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=9)

plt.suptitle('Hypothesis IV: CSF Community Biomarker Validation', fontsize=13)
plt.tight_layout()
plt.savefig('fig_h4_csf_precision.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_h4_csf_precision.png')

## Summary Table: All Hypothesis Results

In [ ]:
print('=' * 80)
print('HYPOTHESIS TESTING SUMMARY')
print('CKG built from 35 ALS CSF proteomics papers (pilot study)')
print('=' * 80)
print()

# Collect results from prior cells (values will be defined if prior cells ran)
summary_rows = []

# Hypothesis I
try:
    bio_bio_n = len(bio_bio)
    bio_gen_n = len(bio_gen)
    a_ratio = bio_bio.mean() / bio_gen.mean() if bio_gen.mean() > 0 else float('nan')
    summary_rows.append({
        'Hypothesis': 'I: Biomarker-biomarker connectivity',
        'Key metric': f'Mean density ratio bio-bio/bio-gen = {a_ratio:.2f}x',
        'n': f'bio-bio pairs={bio_bio_n}, bio-gen pairs={bio_gen_n}'
    })
except Exception:
    summary_rows.append({'Hypothesis': 'I', 'Key metric': 'Not computed', 'n': ''})

# Hypothesis II
try:
    b_ratio = nfl_btw.mean() / other_btw.mean() if other_btw.mean() > 0 else float('nan')
    summary_rows.append({
        'Hypothesis': 'II: NfL hub betweenness',
        'Key metric': f'NfL betweenness {b_ratio:.2f}x higher than other nodes',
        'n': f'NfL nodes={len(nfl_btw)}, other={len(other_btw)}'
    })
except Exception:
    summary_rows.append({'Hypothesis': 'II', 'Key metric': 'Not computed', 'n': ''})

# Hypothesis III
try:
    summary_rows.append({
        'Hypothesis': 'III: Community stability (NMI)',
        'Key metric': f'Mean seed NMI = {mean_seed_nmi:.4f}',
        'n': f'{len(seeds)} seeds tested'
    })
except Exception:
    summary_rows.append({'Hypothesis': 'III', 'Key metric': 'Not computed', 'n': ''})

# Hypothesis IV
try:
    p10 = csf_df.head(10)['is_established'].mean()
    summary_rows.append({
        'Hypothesis': 'IV: CSF biomarker precision',
        'Key metric': f'Precision@10 = {p10:.1%}, Overall = {overall_precision:.1%}',
        'n': f'{len(csf_nodes)} CSF community nodes'
    })
except Exception:
    summary_rows.append({'Hypothesis': 'IV', 'Key metric': 'Not computed', 'n': ''})

print(tabulate(pd.DataFrame(summary_rows), headers='keys', tablefmt='grid', showindex=False))
print()
print('CAVEATS:')
print('  - All results are from a pilot graph of 35 ALS CSF proteomics papers')
print('  - Community sizes are small (6-279 nodes). Statistical tests have low power.')
print('  - Present findings as hypothesis-generating, not confirmatory.')
print('  - Community structure stability (Hypothesis III) is the most important validation to report.')
print('  - Emphasize biological interpretability over statistical significance.')

## Bonus: Community Size Distribution and Graph Overview

In [ ]:
# Publication-quality overview figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Community size bar chart
ax = axes[0, 0]
comm_ids_sorted  = sorted(comm_summary.keys(), key=lambda x: -comm_summary[x]['size'])
comm_sizes_sorted = [comm_summary[c]['size'] for c in comm_ids_sorted]
comm_labels_sorted = [COMMUNITY_LABELS.get(c, f'C{c}')[:20] for c in comm_ids_sorted]
colors_bar = ['#2b6cb0' if community_supertype.get(c) == 'biomarker'
              else '#e53e3e' if community_supertype.get(c) == 'genetic'
              else '#38a169' if community_supertype.get(c) == 'clinical'
              else '#718096'
              for c in comm_ids_sorted]
ax.barh(range(len(comm_ids_sorted)), comm_sizes_sorted, color=colors_bar)
ax.set_yticks(range(len(comm_ids_sorted)))
ax.set_yticklabels(comm_labels_sorted, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Number of nodes')
ax.set_title('Community sizes')
patch_b = mpatches.Patch(color='#2b6cb0', label='Biomarker')
patch_g = mpatches.Patch(color='#e53e3e', label='Genetic')
patch_c = mpatches.Patch(color='#38a169', label='Clinical')
patch_o = mpatches.Patch(color='#718096', label='Other')
ax.legend(handles=[patch_b, patch_g, patch_c, patch_o], fontsize=8, loc='lower right')

# 2. Degree distribution
ax = axes[0, 1]
degrees = [d for _, d in G.degree()]
ax.hist(degrees, bins=50, color='#2b6cb0', alpha=0.8, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Node degree')
ax.set_ylabel('Count')
ax.set_title('Degree distribution (log scale)')
ax.set_yscale('log')
ax.text(0.6, 0.85, f'Nodes: {G.number_of_nodes():,}\nEdges: {G.number_of_edges():,}\nMean degree: {np.mean(degrees):.1f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# 3. Top 20 nodes by betweenness centrality
ax = axes[1, 0]
top20_btw = centrality_df.nlargest(20, 'betweenness').reset_index(drop=True)
colors_btw = ['#e53e3e' if r else '#2b6cb0' for r in top20_btw['is_nfl']]
ax.barh(range(20), top20_btw['betweenness'], color=colors_btw)
ax.set_yticks(range(20))
ax.set_yticklabels([str(n)[:28] for n in top20_btw['node']], fontsize=7)
ax.invert_yaxis()
ax.set_xlabel('Betweenness centrality')
ax.set_title('Top 20 hub nodes (betweenness)')

# 4. Cross-community connectivity summary
ax = axes[1, 1]
if len(cross_df) > 0:
    pair_plot_df = cross_df.groupby('pair_type')['density'].mean().reset_index().sort_values('density', ascending=False)
    ax.bar(range(len(pair_plot_df)), pair_plot_df['density'],
           color=['#2b6cb0', '#e53e3e', '#38a169', '#718096', '#d69e2e'][:len(pair_plot_df)])
    ax.set_xticks(range(len(pair_plot_df)))
    ax.set_xticklabels(pair_plot_df['pair_type'], rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Mean cross-community edge density')
    ax.set_title('Cross-community connectivity by type pair')

plt.suptitle('ALS Causal Knowledge Graph: Structural Overview\n(pilot: 35 CSF proteomics papers)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fig_overview_panel.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_overview_panel.png')

In [ ]:
# Save all results to CSV for reference.
centrality_df.to_csv('centrality_results.csv', index=False)
cross_df.to_csv('cross_community_edges.csv', index=False)
csf_df.to_csv('csf_community_nodes.csv', index=False)
seed_df.to_csv('stability_seed_results.csv', index=False)
res_df.to_csv('stability_resolution_results.csv', index=False)

print('CSV files saved:')
for fname in ['centrality_results.csv', 'cross_community_edges.csv',
              'csf_community_nodes.csv', 'stability_seed_results.csv',
              'stability_resolution_results.csv']:
    print(f'  {fname}')

print('\nFigures saved:')
for fname in ['fig_community_graph.png', 'fig_h1_cross_community.png',
              'fig_h2_betweenness.png', 'fig_h3_stability.png',
              'fig_h4_csf_precision.png', 'fig_overview_panel.png']:
    print(f'  {fname}')

print('\nDone.')